# Smog City — Week 7: Dimensionality Reduction using PCA

**Dataset:** `aqi_india_38cols_knn_final.csv`

## Aim
Use Principal Component Analysis (PCA) to summarize correlated numeric measurements in fewer dimensions and visualize explained variance.

**Important:** PCA is run on numeric features only, after removing columns that are entirely missing or constant and imputing remaining missing numeric values with the median. The dataset is sampled when it is very large to keep Colab memory use manageable. This is an exploratory analysis, not a predictive-accuracy result.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from google.colab import files
from IPython.display import display

print("Week 7 — PCA")

In [ ]:
uploaded = files.upload()
filename = next(iter(uploaded))
df = pd.read_csv(filename, low_memory=False)
df.columns = df.columns.astype(str).str.strip().str.lower()
print("Loaded:", filename, "| shape:", df.shape)
display(df.head())

In [ ]:
# 1. Select numeric columns, exclude identifiers/calendar fields where recognizable
numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
exclude_terms = ["year", "month", "day", "hour", "minute", "second", "id", "zip", "postal", "latitude", "longitude", "lat", "lon"]
feature_cols = [c for c in numeric_cols if not any(term == c or c.endswith("_" + term) for term in exclude_terms)]

# Drop all-empty and constant features
feature_df = df[feature_cols].replace([np.inf, -np.inf], np.nan)
feature_df = feature_df.dropna(axis=1, how="all")
feature_df = feature_df.loc[:, feature_df.nunique(dropna=True) > 1]

print("Numeric candidate features:", len(numeric_cols))
print("Features used for PCA:", feature_df.columns.tolist())
if feature_df.shape[1] < 2:
    raise ValueError("PCA needs at least two usable numeric features. Check the dataset columns.")

In [ ]:
# 2. Sample large datasets for memory efficiency
MAX_PCA_ROWS = 100000
if len(feature_df) > MAX_PCA_ROWS:
    feature_df = feature_df.sample(MAX_PCA_ROWS, random_state=42)
    print(f"Using reproducible sample of {MAX_PCA_ROWS:,} rows for PCA.")
else:
    print(f"Using all {len(feature_df):,} available rows for PCA.")

# Impute missing numeric values, then standardize
imputer = SimpleImputer(strategy="median")
imputed = imputer.fit_transform(feature_df)
scaler = StandardScaler()
scaled = scaler.fit_transform(imputed)

print("PCA input matrix:", scaled.shape)

In [ ]:
# 3. Fit PCA and show explained variance
pca = PCA()
scores = pca.fit_transform(scaled)
explained = pca.explained_variance_ratio_
cumulative = np.cumsum(explained)

variance_table = pd.DataFrame({
    "principal_component": [f"PC{i+1}" for i in range(len(explained))],
    "explained_variance_ratio": explained,
    "explained_variance_percent": explained * 100,
    "cumulative_variance_percent": cumulative * 100
})
display(variance_table.round(4))
variance_table.to_csv("Week7_PCA_Explained_Variance.csv", index=False)

In [ ]:
# 4. Scree plot
plt.figure(figsize=(9, 5))
plt.plot(range(1, len(explained) + 1), explained * 100, marker="o")
plt.title("Week 7 — PCA Scree Plot")
plt.xlabel("Principal Component")
plt.ylabel("Explained Variance (%)")
plt.xticks(range(1, len(explained) + 1))
plt.tight_layout()
plt.savefig("Week7_PCA_Scree_Plot.png", dpi=200)
plt.show()

# Cumulative explained variance plot
plt.figure(figsize=(9, 5))
plt.plot(range(1, len(cumulative) + 1), cumulative * 100, marker="o")
plt.axhline(80, linestyle="--", label="80% variance")
plt.axhline(90, linestyle="--", label="90% variance")
plt.title("Week 7 — Cumulative Explained Variance")
plt.xlabel("Number of Principal Components")
plt.ylabel("Cumulative Explained Variance (%)")
plt.legend()
plt.tight_layout()
plt.savefig("Week7_PCA_Cumulative_Variance.png", dpi=200)
plt.show()

In [ ]:
# 5. PC1 vs PC2 scatter plot (colored by PC1 score)
if scores.shape[1] >= 2:
    plt.figure(figsize=(8, 6))
    plt.scatter(scores[:, 0], scores[:, 1], s=8, alpha=0.5)
    plt.title("Week 7 — Data Projected onto PC1 and PC2")
    plt.xlabel(f"PC1 ({explained[0]*100:.1f}% variance)")
    plt.ylabel(f"PC2 ({explained[1]*100:.1f}% variance)")
    plt.tight_layout()
    plt.savefig("Week7_PCA_PC1_PC2.png", dpi=200)
    plt.show()

# Save scores with component labels
score_cols = [f"PC{i+1}" for i in range(scores.shape[1])]
scores_df = pd.DataFrame(scores, columns=score_cols)
scores_df.to_csv("Week7_PCA_Scores.csv", index=False)
print("Saved PCA tables, plots, and scores.")

## Week 7 conclusion
Use the variance table to state how much variance PC1 and PC2 explain, and how many components are needed to reach at least 80% cumulative variance. Describe whether the PC1–PC2 scatter shows visible groupings, without claiming groups unless they are visible.